<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">
            Creación de un framework para recomendar las personas más adecuadas para trabajar en un determinado proyecto de ingeniería del software
        </h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barcelo
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 1: Análisis Exploratorio y Limpieza de Datos)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        El objetivo de esta primera libreta es <strong>ingestar, analizar y depurar</strong> los datos brutos provenientes de la macroencuesta anual de <em>Stack Overflow</em> para el año 2025. Esta es la base de calidad sobre la que se construirá todo el sistema de recomendación <strong>NetSTAFF</strong>.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Carga y filtrado inicial:</strong> Seleccionaremos únicamente las variables (features) que aportan valor al contexto de Recursos Humanos y reclutamiento técnico.</li>
        <li><strong>Análisis Exploratorio (EDA):</strong> Visualizaremos la distribución del mercado tecnológico global, identificando perfiles predominantes, rangos salariales y modelos de trabajo.</li>
        <li><strong>Tratamiento de Calidad:</strong> Ejecutaremos una limpieza rigurosa eliminando outliers, imputando valores faltantes y corrigiendo inconsistencias lógicas (ej. años de experiencia vs. edad).</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 1.1 Carga y configuración inicial

Importamos las herramientas fundamentales para el análisis de datos (`pandas`, `numpy`) y las librerías gráficas (`matplotlib`, `seaborn`, `plotly`, `missingno`) que nos permitirán realizar un diagnóstico visual de nuestro *dataset*. Además, establecemos una seed de aleatoriedad (`seed=42`) para garantizar que nuestros resultados sean 100% reproducibles.

In [ ]:
import os
import sys
import warnings

sys.path.append(os.path.abspath(os.path.join('../..')))

from src.soft_skills import config
from src.soft_skills import utils_soft

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
import squarify
from pywaffle import Waffle
from IPython.display import display
from sklearn import set_config

random_state = seed = 42
warnings.filterwarnings("ignore")
set_config(transform_output="pandas")

# DATOS 2019

In [ ]:
#Datos 2019
cols_interes = [
    "Respondent","CodeRev","CodeRevHrs","WelcomeChange",
    "WorkChallenge","SOVisitTo","SOPartFreq","MgrIdiot",
    "MgrWant","PurchaseHow","PurchaseWhat","Hobbyist",
    "OpenSourcer","BetterLife","Extraversion","SocialMedia",
    "JobSeek","LastHireDate","ResumeUpdate","Employment",
    "JobSat","CareerSat","WorkWeekHrs","WorkPlan",
    "WorkRemote","WorkLoc","ImpSyn","JobFactors"
]

df_softskills = pd.read_csv(config.RAW_DATA_PATH_2019, usecols=cols_interes, index_col="Respondent")

display(df_softskills.head(5))

### 1.3 Diagnóstico estructural rápido

Utilizamos el comando `.info()` combinado con la matriz visual de la librería `missingno` para ver una radiografía estructural completa de los datos recién ingeridos. 

Como podemos observar tanto en el recuento de consola como en el gráfico:

1. **Tipos de datos:** la mayoría de nuestras variables técnicas (lenguajes, bases de datos) se han interpretado correctamente como texto (`object`), mientras que la experiencia y los salarios son numéricos (`float64`).
2. **Déficit de Información:** La matriz revela visualmente una alta presencia de valores nulos en casi todo el conjunto de los encuestados. Destaca especialmente la falta de datos en columnas críticas como el salario (`ConvertedCompYearly`), plataformas o herramientas de IA. 

Este diagnóstico es la prueba empírica de que no podemos alimentar a la red neuronal directamente con este dataset, haciendo absolutamente obligatoria la siguiente fase de imputación y limpieza.

In [ ]:
df_softskills.info()
msno.matrix(df_softskills)

### 1.4 Revisión de nulos y duplicados

En esta celda extraemos el porcentaje de datos incompletos por variable del *dataset*, lo cual es muy importante para poder tomar decisiones objetivas. 

1. **Cuantificación de Nulos:** mediante un gráfico de barras vemos el porcentaje exacto de información faltante, lo que ayuda a establecer umbrales críticos de decisión (por ejemplo, saber matemáticamente si una variable requiere imputación de datos o si su déficit es tan alto que debe ser eliminada por completo).

2. **Fuga de Datos (Duplicados):** vemos que tenemos un total de **5.053 filas duplicadas**. Si introducimos perfiles técnicos repetidos en nuestra futura Red Neuronal, el modelo sufrirá de *Overfitting* (sobreajuste), memorizando a esos candidatos en lugar de aprender a generalizar los perfiles.

In [ ]:
missing_percentage = df_softskills.isnull().mean() * 100
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)

if not missing_percentage.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x=missing_percentage.values, 
        y=missing_percentage.index, 
        palette='Reds_r'
    )
    plt.title('Riesgo de Datos Incompletos: % de Valores Nulos por Variable', fontsize=14, pad=15)
    plt.xlabel('% de Nulos')
    plt.ylabel('Variable')
    for i, valor in enumerate(missing_percentage.values):
        plt.text(valor + 0.5, i, f"{valor:.1f}%", va='center', fontsize=10)
        
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()

num_duplicates = df_softskills.duplicated().sum()
print(f"\nRevisión de integridad: Hay {num_duplicates:,} filas duplicadas en el dataset original.")

### 1.5 Análisis estadístico descriptivo y detección de anomalías

En esta celda calculamos la media, desviación estándar, cuartiles y valores extremos para poder ver cuales son las *Top 5* de categorías más frecuentes para las columnas de texto. 

Este paso es muy importante porque nos cuenta la "historia" de los datos y revela anomalías a simple vista:

1. **El perfil predominante:** Gracias al recuento categórico, podemos observar, entre otras, que la mayoría de los candidatos de este dataset son empleados a tiempo completo.

2. **Detección de Outliers (Valores Atípicos):** Los estadísticos numéricos nos muestran señales de alarma clarísimas. Por ejemplo, en la variable (`WorkWeekHrs`) vemos un valor máximo de 4850h, y un mínimo de 1h. En la variable de horas de revisión del código (`CodeRevHrs`) hay algunos valores que superan a las horas semanales trabajadas. 

Estos valores extremos (intencionados o por error de tipeo en la encuesta) destruirían la distribución de pesos y el aprendizaje de la red. Esto justifica matemáticamente y de cara a negocio la necesidad de aplicar filtros restrictivos (*clipping*) en la siguiente fase de limpieza.

In [ ]:
print("RESUMEN DE VARIABLES NUMÉRICAS:")
display(df_softskills.describe().T.round(2))

print("\n\nRESUMEN DE VARIABLES CATEGÓRICAS (Top 5 más frecuentes por columna):")
cat_cols = df_softskills.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    print(f"\n--- {col} ---")
    display(df_softskills[col].value_counts().head(5).to_frame(name='Conteo'))

### 1.6 Realizamos limpieza estricta

En esta celda aplicamos nuestra función `limpieza_datos`del módulo `utils`. Esta función realiza las siguientes tareas:

1. **Eliminación de fugas:** Borra los más de 5.000 perfiles duplicados que hemos visto antes.
2. **Filtrado de calidad:** Descarta candidatos "vacíos" que no hayan respondido a las preguntas clave sobre sus herramientas y tecnologías.
3. **Imputación Inteligente (Flags):** En lugar de borrar a los usuarios que no indicaron sus años de experiencia o herramientas, la función rellena los nulos con ceros y crea columnas bandera (ej. `WorkExp_is_missing`, visibles en la tabla resultante). Esto permite a la red neuronal entender que un cero no significa "cero años de experiencia", sino "el usuario no quiso facilitar este dato", capturando así un patrón de comportamiento sin perder al candidato.

*Resultado:* Hemos logrado retener a **44.061 candidatos** limios, formando un bloque sólido sobre el que construir el algoritmo NetSTAFF.

In [ ]:
df_clean = utils_soft.limpieza_soft_skills_2019(df_softskills)
print(f"Transformaciones aplicadas. Candidatos mantenidos: {df_clean.shape[0]}")
display(df_clean.head(3))